In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, roc_auc_score
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")

In [ ]:
features = pd.read_csv("../data/customer_features.csv")
train    = pd.read_csv("../data/customer_clv_train.csv")
df = train.merge(features, on="cust_id", how="left")

In [ ]:
# fixed 3-way split
df_trainval, df_val = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_thresh = train_test_split(df_trainval, test_size=0.25, random_state=42)

feature_cols = joblib.load("../models/feature_columns.pkl")
X_val = df_val[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
y_val = df_val["revenue_2018_2019"].values

print(f"Val set: {len(df_val)} customers | Features: {len(feature_cols)}")

In [ ]:
# loading models
churn_lgb = joblib.load("../models/churn_lgb_model.pkl")
churn_xgb = joblib.load("../models/churn_xgb_model.pkl")
churn_cat = joblib.load("../models/churn_cat_model.pkl")
iso_lgb   = joblib.load("../models/iso_lgb.pkl")
iso_xgb   = joblib.load("../models/iso_xgb.pkl")
iso_cat   = joblib.load("../models/iso_cat.pkl")
rev_model = joblib.load("../models/rev_model_2stage.pkl")
threshold = joblib.load("../models/best_threshold.pkl")

# 2-stage prediction: chrun classifier ensemble + revenue model
def predict_churn_proba(X):
    p_lgb = iso_lgb.transform(churn_lgb.predict_proba(X)[:, 1])
    p_xgb = iso_xgb.transform(churn_xgb.predict_proba(X)[:, 1])
    p_cat = iso_cat.transform(churn_cat.predict_proba(X)[:, 1])
    return (p_lgb + p_xgb + p_cat) / 3

p_return_val = predict_churn_proba(X_val)
log_preds_val = rev_model.predict(X_val)
rev_preds_val = np.maximum(log_preds_val ** 2, 0)
final_pred = np.where(p_return_val < threshold, 0, p_return_val * rev_preds_val)

# mae and spearman 2-stage model alone
mae_val = mean_absolute_error(y_val, final_pred)
corr_val, _ = spearmanr(y_val, final_pred)
print(f"Val MAE: {mae_val:.2f}  |  Spearman: {corr_val:.4f}  |  Threshold: {threshold:.2f}")

In [ ]:
# loading pure regressor - lgb = best performing model
pure_lgb  = joblib.load("../models/pure_lgb.pkl")
best_beta = joblib.load("../models/best_beta_lgb.pkl")

pure_preds_val = np.maximum(pure_lgb.predict(X_val) ** 2, 0)
final_pred = best_beta * pure_preds_val + (1 - best_beta) * final_pred

# mae and spearman blend 2-stage + pure regressor
mae_val = mean_absolute_error(y_val, final_pred)
corr_val, _ = spearmanr(y_val, final_pred)
print(f"Val MAE with blend: {mae_val:.2f}  |  Spearman: {corr_val:.4f}  |  β: {best_beta:.2f}")

### SHAP analyses
#### SHAP churn classifier

In [ ]:
# SHAP on churn ensemble (mean of the 3 models)
print("=== SHAP — Churn Ensemble (average LGB + XGB + CatBoost) ===")
np.random.seed(42)
shap_idx = np.random.choice(len(X_val), size=500, replace=False)
X_shap_churn = pd.DataFrame(
    X_val.values[shap_idx] if hasattr(X_val, 'values') else X_val[shap_idx],
    columns=feature_cols
).apply(pd.to_numeric, errors='coerce').fillna(0)

# SHAP for every model separately
explainer_lgb_clf = shap.TreeExplainer(churn_lgb)
explainer_xgb_clf = shap.TreeExplainer(churn_xgb)
explainer_cat_clf = shap.TreeExplainer(churn_cat)

shap_lgb_clf = explainer_lgb_clf.shap_values(X_shap_churn)
shap_xgb_clf = explainer_xgb_clf.shap_values(X_shap_churn)
shap_cat_clf = explainer_cat_clf.shap_values(X_shap_churn)

# For binary classification, shap_values returns a list — class 1
if isinstance(shap_lgb_clf, list): shap_lgb_clf = shap_lgb_clf[1]
if isinstance(shap_xgb_clf, list): shap_xgb_clf = shap_xgb_clf[1]
if isinstance(shap_cat_clf, list): shap_cat_clf = shap_cat_clf[1]

# Average of three SHAP matrices — exact consistent with ensemble
shap_vals_churn = (shap_lgb_clf + shap_xgb_clf + shap_cat_clf) / 3

# Beeswarm plot
shap.summary_plot(shap_vals_churn, X_shap_churn, max_display=25, show=True)

In [ ]:
# SHAP bar chart churn ensemble
shap.summary_plot(shap_vals_churn, X_shap_churn, plot_type="bar", max_display=25, show=True)

# Table with top 20 features and their mean |SHAP|
mean_abs_shap_churn = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": np.abs(shap_vals_churn).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

# All churn features SHAP ranking
print("\nAlle churn features SHAP ranking:")
print(mean_abs_shap_churn.to_string(index=False))

#### SHAP revenue predictor 2-stage model

In [ ]:
print("=== SHAP — Revenue Model ===")

# only returners for revenue SHAP
df_val_ret = df_val[df_val["revenue_2018_2019"] > 0].copy()
X_val_ret  = df_val_ret[feature_cols]

np.random.seed(42)
shap_idx_rev = np.random.choice(len(X_val_ret), size=min(500, len(X_val_ret)), replace=False)
X_shap_rev = X_val_ret.iloc[shap_idx_rev].apply(pd.to_numeric, errors='coerce').fillna(0)

explainer_rev = shap.TreeExplainer(rev_model)
shap_vals_rev = explainer_rev.shap_values(X_shap_rev)

shap.summary_plot(shap_vals_rev, X_shap_rev, max_display=25, show=True)

In [ ]:
# SHAP bar chart revenue model
shap.summary_plot(shap_vals_rev, X_shap_rev, plot_type="bar", max_display=25, show=True)

In [ ]:
# All revenue features SHAP ranking
mean_abs_shap_rev = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": np.abs(shap_vals_rev).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

print("\nAlle revenue features SHAP ranking:")
print(mean_abs_shap_rev.to_string(index=False))

#### SHAP pure regressor

In [ ]:
# === SHAP — Pure LGB Regressor ===
print("=== SHAP — Pure LGB Regressor (all customers incl. churners) ===")

# Sample 500 customers
np.random.seed(42)
shap_idx_pure = np.random.choice(len(X_val), size=500, replace=False)
X_shap_pure = X_val.iloc[shap_idx_pure]

# SHAP explainer for pure LGB
explainer_pure = shap.TreeExplainer(pure_lgb)
shap_vals_pure = explainer_pure.shap_values(X_shap_pure)

# Beeswarm plot
print("Beeswarm plot — Pure LGB Regressor")
shap.summary_plot(shap_vals_pure, X_shap_pure, max_display=25, show=True)

# Bar chart
print("Bar chart — Pure LGB Regressor")
shap.summary_plot(shap_vals_pure, X_shap_pure, plot_type="bar", max_display=25, show=True)

# Top 20 features tabel
mean_abs_shap_pure = pd.DataFrame({
    "feature": feature_cols,
    "mean_abs_shap": np.abs(shap_vals_pure).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("\nTop 20 features — Pure LGB Regressor:")
print(mean_abs_shap_pure.head(20).to_string(index=False))

In [ ]:
# === Comparison Pure LGB vs Two-Stage Revenue Model ===
print("\n=== Comparison SHAP Rankings: Pure LGB vs Two-Stage Revenue ===")

# Merge both rankings
comparison = mean_abs_shap_pure.rename(
    columns={"mean_abs_shap": "shap_pure"}
).merge(
    mean_abs_shap_rev.rename(columns={"mean_abs_shap": "shap_twostage"}),
    on="feature",
    how="outer"
).fillna(0)

comparison["rank_pure"] = comparison["shap_pure"].rank(ascending=False).astype(int)
comparison["rank_twostage"] = comparison["shap_twostage"].rank(ascending=False).astype(int)
comparison["rank_diff"] = comparison["rank_pure"] - comparison["rank_twostage"]

print(comparison[["feature", "rank_pure", "shap_pure", 
                   "rank_twostage", "shap_twostage", 
                   "rank_diff"]].sort_values("rank_pure").head(20).to_string(index=False))

In [ ]:
# === Correlationmatrix top SHAP features ===
print("=== Spearman Correlationmatrix — Top SHAP Features ===")

# Top features per model
top_churn = mean_abs_shap_churn.head(15)["feature"].tolist()
top_rev = mean_abs_shap_rev.head(15)["feature"].tolist()
top_pure = mean_abs_shap_pure.head(10)["feature"].tolist()

# Combining unique features
top_features = list(dict.fromkeys(top_churn + top_rev + top_pure))
print(f"Totaal unieke features in correlatiematrix: {len(top_features)}")

# Correlationmatrix on full training data
X_corr = df[top_features].apply(pd.to_numeric, errors='coerce').fillna(0)
corr_matrix = X_corr.corr(method="spearman")

# Plot
plt.figure(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 7}
)
plt.title("Spearman Correlationmatrix — Top SHAP Features\n"
          "(churn top 15 + revenue top 15 + pure regressor top 10)",
          fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# Strongly correlated features
print("\n=== Strongly correlated pairs (|r| ≥ 0.70) ===")
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) >= 0.70:
            corr_pairs.append({
                "feature_1": corr_matrix.columns[i],
                "feature_2": corr_matrix.columns[j],
                "spearman_r": round(r, 3)
            })

corr_pairs_df = pd.DataFrame(corr_pairs).sort_values(
    "spearman_r", key=abs, ascending=False
)
print(corr_pairs_df.to_string(index=False))

In [ ]:
# Waterfall plots: one real churner and one real high-value returner
print("=== Waterfall Plots ===")

# SHAP on revenue model for full val set (sample of 500)
np.random.seed(42)
shap_idx_wf = np.random.choice(len(X_val), size=500, replace=False)
X_shap_wf   = pd.DataFrame(
    X_val.values[shap_idx_wf] if hasattr(X_val, 'values') else X_val[shap_idx_wf],
    columns=feature_cols
).apply(pd.to_numeric, errors='coerce').fillna(0)
y_shap_wf   = y_val[shap_idx_wf]
pred_shap_wf = final_pred[shap_idx_wf]

shap_vals_wf = explainer_rev.shap_values(X_shap_wf)

# Find real churner in sample
churner_mask  = y_shap_wf == 0
returner_mask = y_shap_wf > 0

if churner_mask.sum() == 0:
    print("No churners in sample — increase sample size")
else:
    # chose churner with highest predicted revenue
    # (where model thought that he would return but he didn't)
    churner_indices = np.where(churner_mask)[0]
    churner_idx = churner_indices[np.argmax(pred_shap_wf[churner_indices])]
    
    # Chose client with highest true revenue
    highval_idx = np.argmax(y_shap_wf)
    
    print(f"Churner: true revenue = €{y_shap_wf[churner_idx]:.0f}, "
          f"predicted = €{pred_shap_wf[churner_idx]:.0f}")
    print(f"High-value: true revenue = €{y_shap_wf[highval_idx]:.0f}, "
          f"predicted = €{pred_shap_wf[highval_idx]:.0f}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for ax, idx, label in [
        (axes[0], churner_idx,  
         f"Churned customer\n(true €{y_shap_wf[churner_idx]:.0f} | pred €{pred_shap_wf[churner_idx]:.0f})"),
        (axes[1], highval_idx,  
         f"High-value customer\n(true €{y_shap_wf[highval_idx]:.0f} | pred €{pred_shap_wf[highval_idx]:.0f})"),
    ]:
        sv      = shap_vals_wf[idx]
        top_n   = 10
        order   = np.argsort(np.abs(sv))[::-1][:top_n]
        top_sv   = sv[order]
        top_feat = [feature_cols[i] for i in order]
        top_val  = X_shap_wf.iloc[idx].values[order]
        
        colors = ["#d73027" if v > 0 else "#1a9850" for v in top_sv]
        
        bars = ax.barh(range(top_n), top_sv[::-1], 
                       color=colors[::-1], edgecolor="none")
        ax.set_yticks(range(top_n))
        
        # feature name + real value
        labels = [f"{feat} = {val:.2f}" 
                  for feat, val in zip(top_feat[::-1], top_val[::-1])]
        ax.set_yticklabels(labels, fontsize=8)
        ax.axvline(0, color="black", lw=0.8)
        ax.set_xlabel("SHAP value (sqrt revenue space)")
        ax.set_title(label, fontsize=10)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Error analysis
print("=== Error Analysis ===")
err_df = pd.DataFrame({
    "true":      y_val,
    "pred":      final_pred,
    "abs_error": np.abs(final_pred - y_val),
    "churned":   (y_val == 0).astype(int)
})

# MAE churners vs returners
mae_ch = err_df.loc[err_df["churned"] == 1, "abs_error"].mean()
mae_re = err_df.loc[err_df["churned"] == 0, "abs_error"].mean()
print(f"MAE — churners  (revenue = 0):  {mae_ch:.2f}")
print(f"MAE — returners (revenue > 0):  {mae_re:.2f}")
print(f"MAE — total:                   {mae_val:.2f}")

In [ ]:
# === MAE Decomposition: FN / Regressor / FP — Finale Blend ===
# calculated on finale blend (β=0.45 × pure_lgb + 0.55 × two_stage)
# Consistent with revenue tier analyse

print("=== MAE Decomposition — Finale Blend ===")
print(f"β = {best_beta:.2f} | Threshold = {threshold:.2f}")
print()

# Classifications based on churn threshold
is_returner = y_val > 0
is_churner  = y_val == 0
pred_return = p_return_val >= threshold
pred_churn  = p_return_val < threshold

# classification errors
FN_mask = is_returner & pred_churn    # False negatives: real returner, predicted as churner → €0
FP_mask = is_churner  & pred_return   # False positives: real churner, predicted as returner → positive
TP_mask = is_returner & pred_return   # True positives: correctly identified as real returner

# MAE for each group
mae_FN = np.mean(np.abs(y_val[FN_mask] - final_pred[FN_mask]))
mae_FP = np.mean(np.abs(y_val[FP_mask] - final_pred[FP_mask]))
mae_TP = np.mean(np.abs(y_val[TP_mask] - final_pred[TP_mask]))

# Contribution to totale MAE
n = len(y_val)
contribution_FN = np.sum(np.abs(y_val[FN_mask] - final_pred[FN_mask])) / n
contribution_FP = np.sum(np.abs(y_val[FP_mask] - final_pred[FP_mask])) / n
contribution_TP = np.sum(np.abs(y_val[TP_mask] - final_pred[TP_mask])) / n
total_mae = contribution_FN + contribution_FP + contribution_TP

# Context
n_returners = is_returner.sum()
fn_pct_of_returners = FN_mask.sum() / n_returners * 100

print(f"{'Type':<35} {'N':>6} {'MAE':>8} {'Contribution':>14} {'% of total':>10}")
print(f"-" * 77)
print(f"{'False Negatives (returner→0)':<35} {FN_mask.sum():>6} {mae_FN:>8.2f} "
      f"{contribution_FN:>14.2f} {contribution_FN/total_mae*100:>9.1f}%")
print(f"{'False Positives (churner→positive)':<35} {FP_mask.sum():>6} {mae_FP:>8.2f} "
      f"{contribution_FP:>14.2f} {contribution_FP/total_mae*100:>9.1f}%")
print(f"{'Regressor errors (correct ID)':<35} {TP_mask.sum():>6} {mae_TP:>8.2f} "
      f"{contribution_TP:>14.2f} {contribution_TP/total_mae*100:>9.1f}%")
print(f"-" * 77)
print(f"{'Total MAE':<35} {n:>6} {'':>8} {total_mae:>14.2f} {'100.0%':>10}")
print()
print(f"Context:")
print(f"  Total true returners in val set:     {n_returners}")
print(f"  False Negatives as % of returners:   {fn_pct_of_returners:.1f}%")
print(f"  Average true revenue of FN customers: €{y_val[FN_mask].mean():.2f}")
print()
print(f"Verification: {total_mae:.4f} == {mean_absolute_error(y_val, final_pred):.4f}")
print()
print(f"Note: FN/FP classification based on p_return vs threshold {threshold:.2f}")
print(f"      final_pred = {best_beta:.2f} × pure_lgb + {1-best_beta:.2f} × two_stage")

In [ ]:
# tier revenue bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tiers = ["Churners\n(€0)", "€1-€100", "€101-€200", "€201-€500", "€500+"]
tier_mae = [9.32, 58.99, 116.42, 234.81, 545.24]  # finale pipeline waarden

axes[0].bar(tiers, tier_mae, color="steelblue", edgecolor="none")
axes[0].set_ylabel("Mean Absolute Error")
axes[0].set_title("MAE by revenue tier — Final pipeline")

axes[1].scatter(np.log1p(err_df["true"]), np.log1p(err_df["pred"]), alpha=0.15, s=4)
lim = max(np.log1p(err_df["true"]).max(), np.log1p(err_df["pred"]).max())
axes[1].plot([0, lim], [0, lim], "r--", lw=1)
axes[1].set_xlabel("log(True revenue + 1)")
axes[1].set_ylabel("log(Predicted revenue + 1)")
axes[1].set_title(f"Predicted vs True Revenue (MAE {mae_val:.2f})")
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
residuals = final_pred - y_val
plt.figure(figsize=(10, 4))
plt.hist(residuals, bins=100, edgecolor="none", color="steelblue")
plt.axvline(0, color="red", linestyle="--", label="Zero error")
plt.xlabel("Prediction error (predicted − actual)")
plt.ylabel("Count")
plt.title("Residual distributie")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Mean error:   {residuals.mean():.2f}")
print(f"Median error: {np.median(residuals):.2f}")

In [ ]:
# Prediction distribution
plt.figure(figsize=(10, 4))
plt.hist(final_pred[final_pred > 0], bins=80, edgecolor="none", color="steelblue")
plt.xlabel("Predicted CLV (€)")
plt.ylabel("Count")
plt.title(f"Distributie van niet-nul voorspellingen (n={(final_pred > 0).sum()})")
plt.tight_layout()
plt.show()
print(pd.Series(final_pred).describe())